# Code-A-Nova Machine Learning Internship
## Task: Level 1(b) - Email & SMS Spam Detection

### Project Objective
Build a machine learning pipeline that classifies emails / SMS messages as **SPAM** or **NOT SPAM (HAM)** using Natural Language Processing (NLP) techniques.

### Pipeline Steps:
1. **Exploratory Data Analysis (EDA):** Class distribution and missing values.
2. **NLP Preprocessing:** Lowercasing, noise/punctuation removal, tokenization, stopword removal, and stemming.
3. **Feature Extraction:** TF-IDF Vectorization without data leakage.
4. **Model Training & Comparison:** Multinomial Naive Bayes vs. Logistic Regression.
5. **Evaluation:** Accuracy, Precision, Recall, F1-Score, and Confusion Matrix.
6. **Inference:** Testing predictions on new unseen messages.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is in sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
from src.preprocess import load_and_inspect_data, clean_text, preprocess_dataset
from src.features import extract_features
from src.train import split_data, train_and_compare_models
from src.predict import predict_email

print("All modules imported successfully!")

### 1. Load and Inspect the Dataset

In [ ]:
data_path = os.path.join(ROOT_DIR, 'data', 'spam.csv')
df = load_and_inspect_data(data_path)
df.head()

### 2. Preprocessing Walkthrough
Let's observe how raw text is transformed step by step:

In [ ]:
sample_raw = "WINNER! You've been chosen for a $1,000 Walmart Gift Card! Call 0800-123-456 or click http://fake.link now!"
print("Raw Text:      ", sample_raw)
print("Preprocessed:  ", clean_text(sample_raw, use_stemming=True))

# Clean the entire dataset
df_clean = preprocess_dataset(df, text_column='message', use_stemming=True)

### 3. Stratified Train-Test Split (80/20) and TF-IDF Feature Extraction
We fit the TF-IDF vectorizer **strictly** on the training set to prevent data leakage.

In [ ]:
X_train, X_test, y_train, y_test = split_data(df_clean, test_size=0.2, random_state=42)

X_train_vec, X_test_vec, vectorizer = extract_features(
    X_train,
    X_test,
    method='tfidf',
    max_features=5000,
    ngram_range=(1, 2)
)

### 4. Model Training & Comparison

In [ ]:
best_results, best_model = train_and_compare_models(
    X_train_vec,
    X_test_vec,
    y_train,
    y_test
)

### 5. Interactive Testing & Confidence Scoring

In [ ]:
test_messages = [
    "Congratulations! Claim your free £500 Amazon gift card now! Call 09061701461.",
    "Hey, can you send over the slides before tomorrow's team presentation?",
    "URGENT: Your bank account will be suspended. Click here to confirm identity."
]

for msg in test_messages:
    res = predict_email(msg, model=best_model, vectorizer=vectorizer)
    print(f"Message: '{msg[:55]}...'")
    print(f"Prediction: [{res['prediction']}] with {res['confidence']}% confidence\n")